In [2]:
# MOUNT DRIVE
# ===============================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ===============================
# IMPORTS
# ===============================
import cv2
import numpy as np
import os
import pandas as pd
import time
import tracemalloc
import platform
import multiprocessing
from scipy.special import gamma

from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import mean_squared_error as mse
from skimage.measure import shannon_entropy

np.random.seed(42)

# ===============================
# PATHS
# ===============================
image_folder = "/content/drive/MyDrive/Colab Notebooks/data/low_quality_images"
output_folder = "/content/drive/MyDrive/Colab Notebooks/results/output_ICCBF"

os.makedirs(output_folder, exist_ok=True)

# ===============================
# METRICS
# ===============================
def compute_metrics(original, processed):

    original = original.astype(np.float32)
    processed = processed.astype(np.float32)

    ssim_val = ssim(original, processed, data_range=255)
    psnr_val = psnr(original, processed, data_range=255)
    mse_val = mse(original, processed)

    mae_val = np.mean(np.abs(original - processed))
    entropy_val = shannon_entropy(processed)

    edges_orig = cv2.Canny(original.astype(np.uint8), 100, 200)
    edges_proc = cv2.Canny(processed.astype(np.uint8), 100, 200)

    if np.sum(edges_orig) == 0:
        epi_val = 0
    else:
        epi_val = np.sum(edges_orig & edges_proc) / np.sum(edges_orig)

    return ssim_val, psnr_val, mse_val, mae_val, entropy_val, epi_val

def compute_runtime_efficiency(execution_time):
    return 0 if execution_time == 0 else 1 / execution_time

# ===============================
# ENHANCEMENT
# ===============================
def enhance_image(img, clip, tile, d, sigmaColor, sigmaSpace):

    clahe = cv2.createCLAHE(
        clipLimit=float(clip),
        tileGridSize=(int(tile), int(tile))
    )

    cl = clahe.apply(img)

    bf = cv2.bilateralFilter(
        cl,
        int(d),
        sigmaColor,
        sigmaSpace
    )

    return bf

# ===============================
# FITNESS FUNCTION
# ===============================
def get_fitness(img):

    def f(p):

        clip, tile, d, sigmaColor, sigmaSpace = p

        enhanced = enhance_image(
            img,
            clip,
            tile,
            d,
            sigmaColor,
            sigmaSpace
        )

        ssim_val = ssim(img, enhanced, data_range=255)
        psnr_val = psnr(img, enhanced, data_range=255)
        mse_val = mse(img, enhanced)

        return (0.4 * ssim_val) + (0.4 * psnr_val) - (0.2 * mse_val)

    return f

# ===============================
# ICS
# ===============================
class ICS:

    def __init__(self, fitness, bounds, n=10, pa=0.25, beta=1.5, iters=8):

        self.fit = fitness
        self.bounds = np.array(bounds)
        self.n = n
        self.pa = pa
        self.beta = beta
        self.iters = iters
        self.dim = len(bounds)

        self.pop = np.random.uniform(
            self.bounds[:,0],
            self.bounds[:,1],
            (n,self.dim)
        )

        self.fvals = np.array([self.fit(x) for x in self.pop])

    def levy(self):

        b = self.beta

        sigma = (
            gamma(1+b)
            * np.sin(np.pi*b/2)
            / (
                gamma((1+b)/2)
                * b
                * 2**((b-1)/2)
            )
        )**(1/b)

        u = np.random.normal(0, sigma, self.dim)
        v = np.random.normal(0, 1, self.dim)

        return u / (np.abs(v)**(1/b))

    def clip(self, x):

        return np.clip(
            x,
            self.bounds[:,0],
            self.bounds[:,1]
        )

    def run(self):

        for _ in range(self.iters):

            for i in range(self.n):

                new = self.clip(
                    self.pop[i] + 0.03 * self.levy()
                )

                fnew = self.fit(new)

                j = np.random.randint(self.n)

                if fnew > self.fvals[j]:

                    self.pop[j] = new
                    self.fvals[j] = fnew

            for i in range(self.n):

                if np.random.rand() < self.pa:

                    self.pop[i] = np.random.uniform(
                        self.bounds[:,0],
                        self.bounds[:,1],
                        self.dim
                    )

                    self.fvals[i] = self.fit(self.pop[i])

        return self.pop[np.argmax(self.fvals)]

# ===============================
# SYSTEM INFO
# ===============================
system_info = {
    "Processor": platform.processor(),
    "CPU_Cores": multiprocessing.cpu_count(),
    "Platform": platform.platform()
}

print(system_info)

# ===============================
# MAIN LOOP
# ===============================
files = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith((".jpg",".png",".jpeg"))
][:25]

records = []

bounds = [
    (1.0, 3.0),
    (6, 10),
    (5, 9),
    (30, 100),
    (30, 100)
]

for file in files:

    img = cv2.imread(os.path.join(image_folder, file))

    gray = cv2.resize(
        cv2.cvtColor(img, cv2.COLOR_BGR2GRAY),
        (256,256)
    )

    tracemalloc.start()

    start_time = time.perf_counter()

    ics = ICS(get_fitness(gray), bounds)
    best = ics.run()

    enhanced = enhance_image(gray, *best)

    end_time = time.perf_counter()

    current_mem, peak_mem = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    execution_time = end_time - start_time
    memory_usage_mb = peak_mem / (1024 * 1024)
    runtime_efficiency = compute_runtime_efficiency(execution_time)

    cv2.imwrite(
        os.path.join(output_folder, file),
        enhanced
    )

    s, p, m, mae, ent, epi = compute_metrics(
        gray,
        enhanced
    )

    records.append([
        file,
        s,
        p,
        m,
        mae,
        ent,
        epi,
        execution_time,
        memory_usage_mb,
        runtime_efficiency
    ])

# ===============================
# SAVE RESULTS
# ===============================
df = pd.DataFrame(records, columns=[
    "Image",
    "SSIM",
    "PSNR",
    "MSE",
    "MAE",
    "Entropy",
    "EPI",
    "Execution_Time_sec",
    "Memory_MB",
    "Runtime_Efficiency"
])

df.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/results/output_ICCBF/results.csv",
    index=False
)

performance_summary = pd.DataFrame({
    "Metric":[
        "Average Execution Time (s)",
        "Average Memory Usage (MB)",
        "Average Runtime Efficiency"
    ],
    "Value":[
        df["Execution_Time_sec"].mean(),
        df["Memory_MB"].mean(),
        df["Runtime_Efficiency"].mean()
    ]
})

performance_summary.to_csv(
    "/content/drive/MyDrive/Colab Notebooks/results/output_ICCBF/performance_summary.csv",
    index=False
)

print(df.mean(numeric_only=True))
print("FINAL BEST METHOD DONE")


Mounted at /content/drive
{'Processor': 'x86_64', 'CPU_Cores': 2, 'Platform': 'Linux-6.6.122+-x86_64-with-glibc2.35'}
SSIM                    0.908671
PSNR                   24.566041
MSE                   238.639797
MAE                    11.770051
Entropy                 7.581938
EPI                     0.651345
Execution_Time_sec      1.534605
Memory_MB               8.194120
Runtime_Efficiency      0.664322
dtype: float64
FINAL BEST METHOD DONE
